# 03 — Quantizacao no Qdrant

O Qdrant oferece quantizacao nativa que reduz memoria e acelera buscas automaticamente.

## Tipos suportados

| Tipo | Reducao de Memoria | Velocidade | Recall vs float32 |
|------|--------------------|-----------|-------------------|
| Scalar (int8) | 4x | 2x | 97-99% |
| Product (PQ) | 8-32x | 4-8x | 90-95% |
| Binary | 32x | 10-40x | 60-80% |

**Recomendacao para RAG:** Scalar Quantization (int8) — melhor equilibrio.

In [ ]:
from qdrant_client import QdrantClient
from qdrant_client.models import (
    Distance, VectorParams, PointStruct,
    ScalarQuantization, ScalarQuantizationConfig, ScalarType,
    ProductQuantization, ProductQuantizationConfig, CompressionRatio,
    BinaryQuantization, BinaryQuantizationConfig,
    SearchParams, QuantizationSearchParams,
)
from sentence_transformers import SentenceTransformer
import numpy as np
import time
import pandas as pd
import matplotlib.pyplot as plt

client = QdrantClient(host='localhost', port=6333, grpc_port=6334, prefer_grpc=True)
model = SentenceTransformer('all-MiniLM-L6-v2')

# Dataset de teste
np.random.seed(42)
n_docs = 1000
frases_base = [
    'machine learning e uma subarea de inteligencia artificial',
    'bancos de dados relacionais usam SQL para consultas',
    'cloud computing oferece escalabilidade e flexibilidade',
    'seguranca cibernetica protege sistemas contra ataques',
    'frameworks web facilitam desenvolvimento de APIs',
]
docs = [f'{frases_base[i % len(frases_base)]} (variante {i})' for i in range(n_docs)]
all_embs = model.encode(docs, normalize_embeddings=True, show_progress_bar=True)
print(f'{n_docs} docs embedados')

In [ ]:
import random
random.seed(42)
np.random.seed(42)

n_docs = 1000
temas = ['machine learning', 'banco de dados', 'cloud computing', 'seguranca',
         'frontend', 'backend', 'DevOps', 'data science']

docs = [
    f"{random.choice(temas)}: documento {i} sobre {random.choice(['fundamentos', 'avancado', 'pratico'])}"
    for i in range(n_docs)
]

print(f'Criando embeddings para {n_docs} documentos...')
t0 = time.time()
all_embs = model.encode(docs, normalize_embeddings=True, batch_size=128, show_progress_bar=True)
print(f'Shape: {all_embs.shape} | Tempo: {time.time()-t0:.1f}s')


## 3.1 Scalar Quantization (int8) — Recomendado

In [ ]:
def criar_e_indexar(nome, quantization_config):
    if client.collection_exists(nome):
        client.delete_collection(nome)
    client.create_collection(
        collection_name=nome,
        vectors_config=VectorParams(size=384, distance=Distance.COSINE),
        quantization_config=quantization_config,
    )
    points = [
        PointStruct(id=i, vector=all_embs[i].tolist(), payload={'texto': docs[i]})
        for i in range(n_docs)
    ]
    client.upsert(collection_name=nome, points=points)
    print(f'Collection {nome} pronta ({n_docs} docs)')

# Scalar quantization
criar_e_indexar(
    'quant_scalar',
    ScalarQuantization(
        scalar=ScalarQuantizationConfig(
            type=ScalarType.INT8,
            quantile=0.99,   # ignorar 1% dos valores extremos
            always_ram=True, # manter int8 na RAM (mais rapido)
        )
    )
)

# Sem quantizacao (baseline)
criar_e_indexar('quant_none', None)

In [ ]:
def benchmark_recall_speed(collection_name, use_quantization, rescore=True, n_queries=20):
    query_indices = np.random.choice(n_docs, n_queries, replace=False)
    
    search_params = None
    if use_quantization:
        search_params = SearchParams(
            quantization=QuantizationSearchParams(
                ignore=False,
                rescore=rescore,
                oversampling=2.0,
            )
        )
    
    t0 = time.perf_counter()
    all_results = []
    for qi in query_indices:
        results = client.query_points(
            collection_name=collection_name,
            query=all_embs[qi].tolist(),
            limit=10,
            search_params=search_params,
        ).points
        all_results.append([r.id for r in results])
    elapsed = time.perf_counter() - t0
    
    return all_results, elapsed / n_queries * 1000  # ms/query

# Baseline (float32)
results_float32, ms_float32 = benchmark_recall_speed('quant_none', False)

# Int8 com rescore
results_int8_rescore, ms_int8_rescore = benchmark_recall_speed('quant_scalar', True, rescore=True)

# Int8 sem rescore (mais rapido, menor recall)
results_int8_norescore, ms_int8_norescore = benchmark_recall_speed('quant_scalar', True, rescore=False)

# Calcular recall@10 (comparando com float32 como ground truth)
def recall_at_k(ground_truth_lists, pred_lists, k=10):
    recalls = []
    for gt, pred in zip(ground_truth_lists, pred_lists):
        gt_set = set(gt[:k])
        pred_set = set(pred[:k])
        recalls.append(len(gt_set & pred_set) / len(gt_set))
    return np.mean(recalls)

recall_int8_rescore   = recall_at_k(results_float32, results_int8_rescore)
recall_int8_norescore = recall_at_k(results_float32, results_int8_norescore)

print('Resultados do Benchmark:')
print(f'\nfloat32 (baseline):    {ms_float32:.2f}ms/query | Recall@10: 100%')
print(f'int8 + rescore:        {ms_int8_rescore:.2f}ms/query | Recall@10: {recall_int8_rescore:.1%}')
print(f'int8 sem rescore:      {ms_int8_norescore:.2f}ms/query | Recall@10: {recall_int8_norescore:.1%}')

In [ ]:
# Visualizacao speed vs recall
fig, ax = plt.subplots(figsize=(8, 5))

pontos = [
    (ms_float32, 1.0, 'float32 (baseline)', '#e74c3c'),
    (ms_int8_rescore, recall_int8_rescore, 'int8 + rescore', '#2ecc71'),
    (ms_int8_norescore, recall_int8_norescore, 'int8 sem rescore', '#3498db'),
]

for ms, recall, label, cor in pontos:
    ax.scatter(ms, recall, s=300, color=cor, label=label, zorder=3)
    ax.annotate(f'{label}\n{ms:.1f}ms, {recall:.1%}',
               (ms, recall), xytext=(5, -15), textcoords='offset points', fontsize=9)

ax.axhline(y=0.97, color='gray', linestyle='--', alpha=0.5, label='97% threshold')
ax.set_xlabel('Latencia (ms/query)')
ax.set_ylabel('Recall@10 vs float32 baseline')
ax.set_title('Quantizacao: Speed vs Recall Trade-off', fontsize=13, fontweight='bold')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)
ax.set_ylim(0.8, 1.05)
plt.tight_layout()
plt.show()

## 3.2 Quando usar rescore?

```
rescore=True:
  1. Buscar top-k*oversampling candidatos com int8 (rapido)
  2. Re-calcular scores dos top candidatos com float32 (preciso)
  3. Retornar top-k refinados
  → Melhor recall, um pouco mais lento

rescore=False:
  1. Buscar diretamente com int8
  → Mais rapido, recall um pouco menor
```

**Para RAG:** Use `rescore=True` — a latencia extra e pequena e o recall melhora muito.

## Resumo

| Cenario | Configuracao |
|---------|-------------|
| Desenvolvimento | Sem quantizacao (simplicidade) |
| Producao padrao | Scalar int8 + rescore=True |
| Alta escala (>10M docs) | Product Quantization |
| Busca de imagens em escala web | Binary Quantization |